# 02 — Transformation des données brutes

Ce notebook transforme les CSV bruts de `data/raw/` en **14 DataFrames** normalisés (3NF),
prêts à être chargés dans DuckDB.

**Pipeline ETL :**
```
01_extract              Kaggle → data/raw/*.csv
02_transform (ici)      data/raw/*.csv → 14 DataFrames (3NF)
03_load                 DataFrames → DuckDB via SQLAlchemy + Alembic
```

## 1. Chargement des CSV bruts

In [ ]:
from src.etl.transform.config import (
    load_main,
    load_players_db,
    load_games_by_players,
    load_games_by_teams,
)

df_main = load_main()
df_players_db = load_players_db()
df_gbp = load_games_by_players()
df_gbt = load_games_by_teams()

print(f'main: {len(df_main):,} lignes')
print(f'players_db: {len(df_players_db):,} lignes')
print(f'games_by_players: {len(df_gbp):,} lignes')
print(f'games_by_teams: {len(df_gbt):,} lignes')

## 2. Référentiels (Country, Region, Map, Car)

In [ ]:
from src.etl.transform.core import build_countries, build_regions, build_maps, build_cars

df_country = build_countries(df_players_db)
df_region = build_regions(df_main, df_gbt)
df_map = build_maps(df_main)
df_car = build_cars(df_gbp)

print(f'Country: {len(df_country)} | Region: {len(df_region)} | Map: {len(df_map)} | Car: {len(df_car)}')
display(df_region)
display(df_map)
display(df_car)

## 3. Entités (Player, Team)

In [ ]:
from src.etl.transform.core import build_players, build_teams

df_player = build_players(df_players_db)
df_team = build_teams(df_gbt)

print(f'Player: {len(df_player):,} | Team: {len(df_team)}')
display(df_player.head())
display(df_team.head())

## 4. Hiérarchie (Event → Stage → Match → Game)

In [ ]:
from src.etl.transform.core import build_events, build_stages, build_matches, build_games

df_event = build_events(df_main)
df_stage = build_stages(df_main)
df_match = build_matches(df_main)
df_game = build_games(df_main)

print(f'Event: {len(df_event)} | Stage: {len(df_stage)} | Match: {len(df_match):,} | Game: {len(df_game):,}')
display(df_event.head())
display(df_stage.head())
display(df_game.head())

## 5. Participation (GamePlayer, GameTeam)

In [ ]:
from src.etl.transform.core import build_game_players, build_game_teams

df_game_player = build_game_players(df_gbp)
df_game_team = build_game_teams(df_gbt)

print(f'GamePlayer: {len(df_game_player):,} | GameTeam: {len(df_game_team):,}')
display(df_game_player.head())
display(df_game_team.head())

## 6. Stats EAV (StatType + Stat)

In [ ]:
from src.etl.transform.core import build_stat_types, build_stats

df_stat_type = build_stat_types()
print(f'StatType: {len(df_stat_type)} types')
display(df_stat_type.groupby('category').count())

In [ ]:
df_stat = build_stats(df_gbp, df_gbt)
print(f'Stat: {len(df_stat):,} lignes')
display(df_stat.head(10))

## 7. Export vers data/processed/

In [ ]:
from src.etl.transform.core import export_tables

tables = {
    'country': df_country, 'region': df_region, 'map': df_map, 'car': df_car,
    'player': df_player, 'team': df_team,
    'event': df_event, 'stage': df_stage, 'match': df_match, 'game': df_game,
    'game_player': df_game_player, 'game_team': df_game_team,
    'stat_type': df_stat_type, 'stat': df_stat,
}

written = export_tables(tables)
for p in written:
    print(f'  -> {p.name}')
print(f'\n{len(written)} fichiers CSV ecrits dans data/processed/')

## 8. Résumé

In [ ]:
import pandas as pd

summary = pd.DataFrame([
    {'table': name, 'lignes': len(df), 'colonnes': len(df.columns)}
    for name, df in tables.items()
])
display(summary)
print(f'\nTotal : {summary["lignes"].sum():,} lignes dans {len(tables)} tables')
print(f'\nProchaine étape : 03_load.ipynb')